# Aligning Language Models with DPO and Reward Modeling

Pretraining and SFT teach the model to generate text and follow instructions. Preference optimization shapes [*which*]{.underline} instructions it follows well — aligning model behavior to human preferences at scale. This notebook covers two complementary approaches: **DPO** (Direct Preference Optimization), which eliminates the need for an explicit reward model, and **reward model training**, which provides the scalar signal needed for GRPO in the next notebook.

## Part 1: Direct Preference Optimization

The standard RLHF pipeline is a three-stage procedure: (1) supervised fine-tuning, (2) reward model training, and (3) policy optimization with PPO. DPO [@rafailov2023] collapses stages (2) and (3) into a single supervised objective — no explicit reward model required, no reinforcement learning loop.

## The RLHF Objective

Let $\pi_\theta$ be the policy (the language model) and $\pi_{\text{ref}}$ be a frozen reference policy (a copy of the SFT model). The RLHF objective adds a KL penalty to prevent the policy from drifting too far from the reference and **reward hacking** — exploiting the reward model with degenerate outputs:

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot \mid x)} \left[ r(x, y) \right] - \beta \, D_{\text{KL}}\!\left(\pi_\theta(\cdot \mid x) \,\|\, \pi_{\text{ref}}(\cdot \mid x)\right)$$

Here $\beta > 0$ controls the strength of the KL penalty: small $\beta$ allows more divergence from the reference, large $\beta$ keeps the policy close to the SFT model. Optimizing this objective directly via PPO requires four models to reside in memory simultaneously: the policy being trained, the reference policy, the reward model, and the value model.

:::{.callout-note}
PPO requires 4 models simultaneously: the policy, reference policy, reward model, and value model. DPO's key contribution is showing these can be collapsed into a single supervised objective.

:::

## The DPO Derivation

The RLHF objective has a [closed-form optimal policy]{.mark}. Setting the functional derivative to zero and solving gives:

$$\pi^*(y \mid x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{1}{\beta} r(x, y)\right)$$

where $Z(x) = \sum_y \pi_{\text{ref}}(y \mid x) \exp(r(x, y) / \beta)$ is a normalizing partition function. Rearranging, we can express the reward in terms of the optimal policy:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x).$$

Now suppose we have human preference data: pairs $(y_w, y_l)$ for prompt $x$ where $y_w$ is the preferred ("won") response and $y_l$ is the dispreferred ("lost") response. Under the **Bradley-Terry model**, the probability that $y_w$ is preferred over $y_l$ is:

$$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l)).$$

Substituting the reward reparameterization and taking the difference:

$$r(x, y_w) - r(x, y_l) = \beta \log \frac{\pi^*(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi^*(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}.$$

The $\beta \log Z(x)$ terms cancel because $y_w$ and $y_l$ share the same prompt $x$. Replacing $\pi^*$ with our parameterized policy $\pi_\theta$ and maximizing the log-likelihood of the observed preferences gives the **DPO loss**:

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\left[\log \sigma\!\left(\beta \left(\log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right)\right]}$$

The quantity $\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ is the **implicit reward** — no separate reward model is required.

:::{.callout-important}
The $Z(x)$ cancellation is the entire trick. It works because both $y_w$ and $y_l$ share the same prompt $x$. If they had different prompts, $Z(x)$ would not cancel and the derivation breaks down.

:::

## Computing Log-Probabilities

The DPO loss requires the log-probability of a complete response $y$ under a model $\pi$, conditioned on a prompt $x$. By the chain rule:

$$\log \pi(y \mid x) = \sum_{t=1}^{|y|} \log \pi(y_t \mid x, y_{<t}).$$

We run the full sequence (prompt + response) through the model to get logits, then sum the log-probabilities of the response tokens — masking prompt positions with `−100`, the same convention used in SFT.

Implementing `sequence_log_probs`:

In [ ]:
import torch
import torch.nn.functional as F

def sequence_log_probs(
    model,
    input_ids:  torch.Tensor,   # (B, T) full sequence: prompt + response
    labels:     torch.Tensor,   # (B, T) response tokens; -100 at prompt positions
) -> torch.Tensor:
    """
    Compute the sum of log-probabilities for each response in the batch.
    Returns: log_probs: (B,) — sum of log p(token) for each response token.
    """
    with torch.no_grad() if not model.training else torch.enable_grad():  # <1>
        logits, _ = model(input_ids)   # (B, T, V)

    logits_shifted = logits[:, :-1, :]      # (B, T-1, V)  # <2>
    labels_shifted = labels[:, 1:]          # (B, T-1)

    response_mask  = (labels_shifted != -100).float()       # <3>

    labels_clipped = labels_shifted.clone()
    labels_clipped[labels_shifted == -100] = 0              # <4>

    log_probs_all = F.log_softmax(logits_shifted, dim=-1)
    token_log_probs = log_probs_all.gather(
        dim=2,
        index=labels_clipped.unsqueeze(2)
    ).squeeze(2)                                            # <5>

    return (token_log_probs * response_mask).sum(dim=1)     # <6>

1. The reference model is called with `torch.no_grad()`; the policy model is called with gradients enabled. We handle both cases by checking `model.training`.
2. The standard language modeling shift: logits at position $t$ predict token at position $t+1$.
3. Response positions are where `labels != -100`. The mask is `float` for subsequent multiplication.
4. Replace `-100` with `0` before indexing into the vocabulary dimension — out-of-range indices would raise an error. The mask ensures these positions contribute nothing.
5. `gather` picks the log-probability of the actual next token at each position, giving shape `(B, T-1)`.
6. Multiply by the response mask to zero out prompt positions, then sum over time to get a scalar per sequence.

## The DPO Loss

With `sequence_log_probs` in hand, computing the DPO loss is straightforward. We compute implicit rewards for chosen and rejected responses under both the policy and the reference, then take the log-sigmoid of the margin.

Implementing `dpo_loss`:

In [ ]:
def dpo_loss(
    policy_chosen_logps:    torch.Tensor,
    policy_rejected_logps:  torch.Tensor,
    ref_chosen_logps:       torch.Tensor,
    ref_rejected_logps:     torch.Tensor,
    beta:                   float = 0.1,
) -> tuple[torch.Tensor, dict]:
    """DPO loss."""
    chosen_reward   = beta * (policy_chosen_logps   - ref_chosen_logps)    # <1>
    rejected_reward = beta * (policy_rejected_logps - ref_ref_logps)       # <2>
    reward_margin   = chosen_reward - rejected_reward                      # <3>
    loss = -F.logsigmoid(reward_margin).mean()                             # <4>
    metrics = {
        'loss':              loss.item(),
        'reward_margin':     reward_margin.mean().item(),
        'chosen_reward':     chosen_reward.mean().item(),
        'rejected_reward':   rejected_reward.mean().item(),
        'accuracy':          (reward_margin > 0).float().mean().item(),    # <5>
    }
    return loss, metrics

1. The implicit reward $\hat{r}(x, y_w) = \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)}$ for the chosen response.
2. Same for the rejected response.
3. The reward margin is the difference of implicit rewards: $\hat{r}_w - \hat{r}_l.$ A positive margin means the policy correctly prefers the chosen response.
4. `F.logsigmoid` is numerically stable. Negating turns a maximization of log-likelihood into a minimization.
5. Preference accuracy: fraction of pairs where the policy assigns a higher implicit reward to the chosen response. At initialization (with an untrained policy equal to the reference), this is $0.5.$

<br>

**The $\beta$ parameter.** $\beta$ controls how aggressively the policy is allowed to diverge from the reference. Typical choices: $\beta = 0.1$ (standard), $\beta = 0.01$ (aggressive, larger updates), $\beta = 0.5$ (conservative, stays close to the SFT model). Larger $\beta$ means the implicit reward signal is smaller in magnitude, producing smaller gradient steps.

## Preference Data

DPO requires a dataset of preference pairs. Each sample is a triple $(x, y_w, y_l)$: a prompt, a preferred response, and a dispreferred response. We store these as JSONL where each line is `{"prompt": "...", "chosen": "...", "rejected": "..."}`.

The `DPODataset` class encodes each sample into two full sequences (prompt + chosen, prompt + rejected) with labels masking the prompt positions:

In [ ]:
from torch.utils.data import Dataset
import json

class DPODataset(Dataset):
    """
    Each line: {"prompt": "...", "chosen": "...", "rejected": "..."}
    Returns: chosen_ids, chosen_labels, rejected_ids, rejected_labels
    """
    def __init__(self, data_path, tokenizer, max_length=512):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = []
        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try: self.samples.append(json.loads(line))
                except json.JSONDecodeError: continue
        print(f"DPODataset: {len(self.samples)} preference pairs")

    def _encode_sample(self, prompt, response):             # <1>
        prompt_text   = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n"
                         f"{SPECIAL_TOKENS['end']}\n"
                         f"{SPECIAL_TOKENS['assistant']}\n")
        response_text = f"{response.strip()}{SPECIAL_TOKENS['end']}\n"
        prompt_ids    = self.tokenizer.encode(prompt_text)
        response_ids  = self.tokenizer.encode(response_text)
        max_resp = self.max_length - len(prompt_ids) - 1
        response_ids = response_ids[:max_resp]              # <2>
        input_ids = prompt_ids + response_ids
        labels    = ([-100] * len(prompt_ids)) + response_ids  # <3>
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long),
        )

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        chosen_ids,   chosen_labels   = self._encode_sample(s['prompt'], s['chosen'])
        rejected_ids, rejected_labels = self._encode_sample(s['prompt'], s['rejected'])
        return chosen_ids, chosen_labels, rejected_ids, rejected_labels


def collate_dpo(batch):
    chosen_ids, chosen_labels, rejected_ids, rejected_labels = zip(*batch)

    def pad(tensors, pad_value):                            # <4>
        max_len = max(t.size(0) for t in tensors)
        out = torch.full((len(tensors), max_len), pad_value, dtype=torch.long)
        for i, t in enumerate(tensors):
            out[i, :t.size(0)] = t
        return out

    return (
        pad(chosen_ids, 0), pad(chosen_labels, -100),
        pad(rejected_ids, 0), pad(rejected_labels, -100),
    )

1. `_encode_sample` builds the full prompt+response sequence using the chat template, then produces labels where prompt positions are masked to `-100`.
2. Response tokens are truncated to fit within `max_length`, preserving the prompt in full.
3. Labels are `-100` for all prompt positions and equal to the token ids for response positions. This matches the `ignore_index` convention used in `sequence_log_probs`.
4. Sequences in a batch are padded to the longest sequence. Padding `input_ids` use `0`; padding `labels` use `-100` so padded positions are masked automatically.

<br>

**Generating a toy DPO dataset.** For the nano model we construct preference pairs from Shakespeare: the chosen response is a real passage, and the rejected response is the same words shuffled into incoherent order:

In [ ]:
from pathlib import Path

def make_toy_dpo_dataset(raw_text, output_path, n_samples=300):
    """Chosen = real Shakespeare passage; rejected = same words shuffled."""
    import random
    random.seed(42)
    words  = raw_text.split()
    chunks = [' '.join(words[i:i+60]) for i in range(0, len(words)-60, 60)]
    prompts = [
        "Write a short passage in the style of Shakespeare.",
        "Continue this dramatic scene.",
        "Write a soliloquy.",
        "Write dialogue for a play.",
    ]
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            chosen       = chunks[i]
            words_chunk  = chosen.split()
            random.shuffle(words_chunk)
            rejected = ' '.join(words_chunk)
            sample   = {'prompt': random.choice(prompts), 'chosen': chosen, 'rejected': rejected}
            f.write(json.dumps(sample) + '\n')
    print(f"Wrote {min(n_samples, len(chunks))} preference pairs to {output_path}")

## The DPO Training Loop

DPO requires two forward passes per batch: one through the policy (which receives gradients) and one through the frozen reference model (no gradients). We run them sequentially to keep peak memory low — the reference forward pass uses `torch.no_grad()`.

The `dpo_train` function loads an SFT checkpoint, optionally injects LoRA, creates a frozen copy of the SFT model as the reference, and trains via the DPO loss. It depends on utilities from [NB01](/courses/llm/01-gpt-architecture.html) (`GPT`, `NanoGPTConfig`), [NB02](/courses/llm/02-tokenization.html) (`Tokenizer`), and [NB08](/courses/llm/08-sft-lora.html) (`inject_lora`, `freeze_base_model`, `make_cosine_schedule`).

In [ ]:
import copy
import numpy as np
from torch.utils.data import DataLoader

def dpo_train(
    sft_model_path, data_path, output_dir,
    beta=0.1, max_lr=5e-5, min_lr=5e-6, warmup_steps=50, max_steps=500,
    batch_size=2, max_length=256, eval_every=100, use_lora=True, lora_rank=8,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if device.type == 'cuda' else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    from tutorial_02 import GPT, NanoGPTConfig
    from tutorial_03 import Tokenizer

    config = NanoGPTConfig()
    tok    = Tokenizer.load('nano_tokenizer.json')

    # --- Policy model ---
    policy = GPT(config).to(device)
    ckpt   = torch.load(sft_model_path, map_location=device)
    policy.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)

    if use_lora:
        inject_lora(policy, rank=lora_rank)          # <1>
        freeze_base_model(policy)
        trainable_params = [p for p in policy.parameters() if p.requires_grad]
    else:
        trainable_params = list(policy.parameters())

    # --- Reference model (frozen copy of SFT) ---
    ref_model = GPT(config).to(device)               # <2>
    ref_model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    print(f"Policy trainable params: {sum(p.numel() for p in trainable_params):,}")
    print(f"Reference model: frozen")

    ds = DPODataset(data_path, tok, max_length=max_length)
    val_size  = max(1, len(ds) // 10)
    train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds) - val_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_dpo, num_workers=1)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_dpo, num_workers=1)

    optimizer = torch.optim.AdamW(trainable_params, lr=max_lr, weight_decay=0.01)
    scheduler = make_cosine_schedule(optimizer, max_lr, min_lr, warmup_steps, max_steps)

    policy.train()
    train_iter = iter(train_loader)
    history    = []

    for step in range(max_steps):
        try:
            chosen_ids, chosen_labels, rejected_ids, rejected_labels = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            chosen_ids, chosen_labels, rejected_ids, rejected_labels = next(train_iter)

        chosen_ids     = chosen_ids.to(device);     chosen_labels     = chosen_labels.to(device)
        rejected_ids   = rejected_ids.to(device);   rejected_labels   = rejected_labels.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            policy_chosen_logps   = sequence_log_probs(policy, chosen_ids, chosen_labels)
            policy_rejected_logps = sequence_log_probs(policy, rejected_ids, rejected_labels)
            with torch.no_grad():                    # <3>
                ref_chosen_logps   = sequence_log_probs(ref_model, chosen_ids, chosen_labels)
                ref_rejected_logps = sequence_log_probs(ref_model, rejected_ids, rejected_labels)
            loss, metrics = dpo_loss(
                policy_chosen_logps, policy_rejected_logps,
                ref_chosen_logps, ref_rejected_logps, beta=beta,
            )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        scheduler.step()

        history.append(metrics)

        if step % 50 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"step {step:4d}  loss={metrics['loss']:.4f}  margin={metrics['reward_margin']:.4f}  acc={metrics['accuracy']:.2%}  lr={lr:.2e}")

        if step % eval_every == 0 and step > 0:
            policy.eval()
            eval_metrics = []
            with torch.no_grad():
                for c_ids, c_lab, r_ids, r_lab in val_loader:
                    c_ids = c_ids.to(device); c_lab = c_lab.to(device)
                    r_ids = r_ids.to(device); r_lab = r_lab.to(device)
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        p_clog = sequence_log_probs(policy, c_ids, c_lab)
                        p_rlog = sequence_log_probs(policy, r_ids, r_lab)
                        rc_log = sequence_log_probs(ref_model, c_ids, c_lab)
                        rr_log = sequence_log_probs(ref_model, r_ids, r_lab)
                        _, m   = dpo_loss(p_clog, p_rlog, rc_log, rr_log, beta=beta)
                    eval_metrics.append(m)
            policy.train()

            avg = {k: np.mean([m[k] for m in eval_metrics]) for k in eval_metrics[0]}
            print(f"  [eval] loss={avg['loss']:.4f}  margin={avg['reward_margin']:.4f}  acc={avg['accuracy']:.2%}")

            torch.save({
                'step': step, 'policy': policy.state_dict(),
                'metrics': avg, 'beta': beta,
            }, f'{output_dir}/dpo_step{step:04d}.pt')

    return policy, history

1. `inject_lora` and `freeze_base_model` from [NB08](/courses/llm/08-sft-lora.html) replace the query/value projections with LoRA adapters and freeze all non-LoRA parameters. This reduces trainable parameters from ~30M to ~500K.
2. The reference model is a separate `GPT` instance loaded from the same SFT checkpoint. It is never updated — its purpose is to anchor the implicit reward baseline.
3. The reference forward passes are always wrapped in `torch.no_grad()`, regardless of the outer `autocast` context, to avoid storing activations for a model we will never backpropagate through.

## Monitoring DPO Training

Four metrics tell the story of a DPO run:

- **Reward margin** ($\hat{r}_w - \hat{r}_l$): should start near $0$ and increase. A negative or declining margin indicates the policy is moving in the wrong direction.
- **Preference accuracy**: fraction of pairs where the margin is positive. Starts at $0.5$, target $0.8$–$0.9$. Stuck at $0.5$ means the model is not learning; reaching $1.0$ too quickly suggests memorization.
- **Chosen reward** ($\hat{r}_w$): should increase as the policy learns to assign higher probability to preferred responses.
- **Rejected reward** ($\hat{r}_l$): should decrease. [Both rewards increasing together]{.mark} is a warning sign — it indicates the policy is diverging uniformly from the reference rather than discriminating between good and bad responses. Remedy: lower $\beta$ or the learning rate.

Visualization of DPO training diagnostics:

In [ ]:
#| code-fold: true
def plot_dpo_training(history: list[dict]):
    import matplotlib.pyplot as plt

    steps = list(range(len(history)))
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle('DPO Training Diagnostics', fontsize=14, fontweight='bold')

    axes[0][0].plot(steps, [m['loss'] for m in history], color='#F44336')
    axes[0][0].set_title('DPO Loss'); axes[0][0].set_xlabel('Step')

    axes[0][1].plot(steps, [m['reward_margin'] for m in history], color='#4CAF50')
    axes[0][1].axhline(0, color='gray', linestyle='--', lw=0.8)
    axes[0][1].set_title('Reward Margin (chosen - rejected)'); axes[0][1].set_xlabel('Step')

    axes[1][0].plot(steps, [m['chosen_reward'] for m in history], color='#2196F3', label='chosen')
    axes[1][0].plot(steps, [m['rejected_reward'] for m in history], color='#FF9800', label='rejected')
    axes[1][0].axhline(0, color='gray', linestyle='--', lw=0.8)
    axes[1][0].set_title('Implicit Rewards'); axes[1][0].legend(); axes[1][0].set_xlabel('Step')

    axes[1][1].plot(steps, [m['accuracy'] for m in history], color='#9C27B0')
    axes[1][1].axhline(0.5, color='gray', linestyle='--', lw=0.8, label='random')
    axes[1][1].set_ylim(0, 1); axes[1][1].set_title('Preference Accuracy')
    axes[1][1].legend(); axes[1][1].set_xlabel('Step')

**Figure.** Four-panel DPO training diagnostic. Top-left: DPO loss should decrease smoothly. Top-right: reward margin should be positive and growing. Bottom-left: chosen reward should rise while rejected reward falls — divergence in opposite directions indicates discrimination is being learned. Bottom-right: preference accuracy should rise from $0.5$ toward $0.8$–$0.9$.

## Part 2: Reward Model Training

DPO dispenses with the reward model entirely. But a separate, explicit reward model is still needed for GRPO — the group relative policy optimization algorithm covered in the next notebook — where the reward signal is evaluated online during training rather than precomputed from preference data. This section covers how to train a [scalar reward function]{.mark} from the same preference pairs used for DPO.

## The Bradley-Terry Model

We model human preferences probabilistically. Given a prompt $x$ and two responses $y_w$ (preferred) and $y_l$ (dispreferred), we assign scalar rewards $r_w = r(x, y_w)$ and $r_l = r(x, y_l)$ and define the probability that $y_w$ is preferred under the **Bradley-Terry model**:

$$P(y_w \succ y_l \mid x) = \sigma(r_w - r_l) = \frac{e^{r_w}}{e^{r_w} + e^{r_l}}.$$

Given a dataset $\mathcal{D}$ of preference pairs, we maximize the log-likelihood:

$$\mathcal{L} = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma(r_w - r_l) \right].$$

This is structurally identical to binary cross-entropy with target $1$ and logit equal to the reward gap $r_w - r_l$.[^bt] It is zero when $r_w \gg r_l$, and approaches $\log 2 \approx 0.693$ when $r_w = r_l.$

[^bt]: The Bradley-Terry model is exactly logistic regression on reward differences. The reward function $r$ is the "weight" being learned, and the feature is the pair $(y_w, y_l)$ for prompt $x$.

## Architecture

A reward model is a language model backbone with a **scalar head**: a single linear layer mapping the final hidden state to a scalar reward $r \in \mathbb{R}$. We extract the hidden state at the [last real token position]{.underline} of the sequence — in a causal transformer, this position has attended to every preceding token and therefore encodes the entire context.

Reusing a pretrained backbone is important: the backbone already knows language, so it can meaningfully distinguish good responses from bad ones after fine-tuning on far fewer preference pairs (typically $10$K–$100$K) than would be needed to train from scratch.

Adding `get_hidden_states` to the `GPT` class:

In [ ]:
# In the GPT class from NB01 — add this method:

def get_hidden_states(self, idx: torch.Tensor) -> torch.Tensor:
    """Forward pass returning hidden states (before lm_head). Shape: (B, T, d_model)"""
    B, T = idx.shape
    assert T <= self.config.max_seq_len
    pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
    tok_emb = self.token_embedding(idx)
    pos_emb = self.position_embedding(pos)
    x = self.dropout(tok_emb + pos_emb)
    for block in self.blocks:
        x = block(x)
    x = self.ln_f(x)
    return x

The `RewardModel` class wraps the GPT backbone with the scalar head and implements score extraction at the last real token position:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    """LM backbone with scalar reward head. Score at last real token."""

    def __init__(self, config, backbone_path=None):
        super().__init__()
        self.backbone    = GPT(config)
        d_model          = config.d_model
        self.reward_head = nn.Linear(d_model, 1, bias=False)   # <1>
        nn.init.zeros_(self.reward_head.weight)                 # <2>

        if backbone_path is not None:
            ckpt  = torch.load(backbone_path, map_location='cpu')
            state = ckpt['model'] if 'model' in ckpt else ckpt
            backbone_state = {k: v for k, v in state.items()
                              if not k.startswith('lm_head')}   # <3>
            missing, unexpected = self.backbone.load_state_dict(
                backbone_state, strict=False
            )
            print(f"Loaded backbone: {len(backbone_state)} tensors")
            if unexpected:
                print(f"  Unexpected keys: {unexpected[:3]}")

    def forward(self, input_ids, attention_mask=None):
        """Returns scalar reward for each sequence. Shape: (B,)"""
        hidden_states = self.backbone.get_hidden_states(input_ids)

        if attention_mask is not None:
            sequence_lengths = attention_mask.sum(dim=1) - 1   # <4>
        else:
            sequence_lengths = torch.full(
                (input_ids.size(0),), input_ids.size(1) - 1,
                dtype=torch.long, device=input_ids.device
            )

        batch_size  = input_ids.size(0)
        last_hidden = hidden_states[                           # <5>
            torch.arange(batch_size, device=input_ids.device),
            sequence_lengths,
        ]
        reward = self.reward_head(last_hidden).squeeze(-1)     # <6>
        return reward

1. A single linear layer mapping `d_model → 1`. No bias: the output is a relative score, not an absolute magnitude.
2. Zero initialization ensures all sequences receive reward $\approx 0$ at the start of training — a neutral baseline.
3. We skip the `lm_head` weights when loading the pretrained checkpoint since the reward head has a different shape and purpose.
4. `attention_mask.sum(dim=1) - 1` gives the index of the last non-padding token in each sequence.
5. Advanced indexing: for each item in the batch, we select the hidden state at its last real token position.
6. `squeeze(-1)` removes the trailing singleton dimension, giving shape `(B,)` rather than `(B, 1)`.

:::{.callout-note}
The reward head is initialized to zero, so initial rewards are $\approx 0$ for all inputs. This is a neutral baseline that avoids any early bias toward positive or negative rewards.

:::

## The Reward Loss

The Bradley-Terry loss for a batch of preference pairs is:

$$\mathcal{L} = -\log \sigma(r_w - r_l) = \log(1 + e^{-(r_w - r_l)}).$$

An optional **margin** parameter forces a minimum gap between chosen and rejected rewards, encouraging the model to produce confidently separated scores rather than learning tiny differences:

$$\mathcal{L}_{\text{margin}} = -\log \sigma(r_w - r_l - m).$$

A typical value is $m = 0.5$–$1.0$.

Implementing `reward_loss`:

In [ ]:
def reward_loss(chosen_rewards, rejected_rewards, margin=0.0):
    """Bradley-Terry ranking loss with optional margin."""
    reward_gap = chosen_rewards - rejected_rewards             # <1>
    if margin > 0:
        loss = -F.logsigmoid(reward_gap - margin).mean()       # <2>
    else:
        loss = -F.logsigmoid(reward_gap).mean()
    metrics = {
        'loss':             loss.item(),
        'reward_gap':       reward_gap.mean().item(),
        'chosen_reward':    chosen_rewards.mean().item(),
        'rejected_reward':  rejected_rewards.mean().item(),
        'accuracy':         (reward_gap > 0).float().mean().item(),  # <3>
    }
    return loss, metrics

1. The reward gap $r_w - r_l$ is the logit of the preference probability under Bradley-Terry.
2. With a margin $m$, the loss only reaches zero when $r_w - r_l > m$ — any gap smaller than $m$ still incurs a positive loss.
3. Preference accuracy: fraction of pairs where the reward model correctly assigns a higher score to the chosen response. This is the primary evaluation metric.

## The Reward Dataset

The `RewardDataset` class uses the same JSONL format as `DPODataset`, but encodes sequences differently: the full conversation text (prompt + response) is encoded as a single flat sequence, and an attention mask tracks which positions are real versus padding.

Implementing `RewardDataset` and `collate_reward`:

In [ ]:
class RewardDataset(Dataset):
    """Returns (chosen_ids, chosen_mask, rejected_ids, rejected_mask) per sample."""

    def __init__(self, data_path, tokenizer, max_length=512):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = []
        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try: self.samples.append(json.loads(line))
                except json.JSONDecodeError: continue
        print(f"RewardDataset: {len(self.samples)} preference pairs")

    def _encode(self, prompt, response):
        text   = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n{SPECIAL_TOKENS['end']}\n"
                  f"{SPECIAL_TOKENS['assistant']}\n{response.strip()}\n{SPECIAL_TOKENS['end']}")
        tokens = self.tokenizer.encode(text)[:self.max_length]
        ids    = torch.tensor(tokens, dtype=torch.long)
        mask   = torch.ones_like(ids)                      # <1>
        return ids, mask

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        c_ids, c_mask = self._encode(s['prompt'], s['chosen'])
        r_ids, r_mask = self._encode(s['prompt'], s['rejected'])
        return c_ids, c_mask, r_ids, r_mask


def collate_reward(batch):
    c_ids, c_masks, r_ids, r_masks = zip(*batch)

    def pad(tensors, pad_value=0):
        max_len = max(t.size(0) for t in tensors)
        out = torch.full((len(tensors), max_len), pad_value, dtype=torch.long)
        for i, t in enumerate(tensors):
            out[i, :t.size(0)] = t
        return out

    return (pad(c_ids), pad(c_masks), pad(r_ids), pad(r_masks))  # <2>

1. The attention mask is all-ones for non-padded tokens. After `collate_reward` pads sequences to the same length, padding positions will have `mask = 0`, which `RewardModel.forward` uses to locate the last real token.
2. Padding `input_ids` use `0`; the attention mask pads with `0` as well (padding positions are masked out during score extraction).

## Training the Reward Model

The reward model training loop has two details that distinguish it from SFT: (1) the reward head uses a $10\times$ higher learning rate than the backbone, since the head is randomly initialized while the backbone is pretrained, and (2) the first `freeze_layers` transformer blocks are frozen to prevent overfitting on small datasets.

Implementing `train_reward_model`:

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, random_split

def train_reward_model(
    backbone_path, data_path, output_dir,
    max_lr=1e-4, min_lr=1e-5, warmup_steps=50, max_steps=1000, batch_size=4,
    max_length=256, margin=0.5, eval_every=100, freeze_layers=4,
):
    """freeze_layers: freeze first N transformer blocks to prevent overfitting."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if device.type == 'cuda' else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    from tutorial_03 import Tokenizer
    tok    = Tokenizer.load('nano_tokenizer.json')
    config = NanoGPTConfig()
    model  = RewardModel(config, backbone_path=backbone_path).to(device)

    for i, block in enumerate(model.backbone.blocks):
        if i < freeze_layers:
            for p in block.parameters():
                p.requires_grad_(False)                    # <1>

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")

    ds       = RewardDataset(data_path, tok, max_length=max_length)
    val_size = max(1, len(ds) // 10)
    train_ds, val_ds = random_split(ds, [len(ds) - val_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_reward, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              collate_fn=collate_reward, num_workers=1)

    optimizer = torch.optim.AdamW([
        {'params': model.reward_head.parameters(),  'lr': max_lr * 10},   # <2>
        {'params': [p for n, p in model.backbone.named_parameters()
                    if p.requires_grad],             'lr': max_lr},
    ], weight_decay=0.01)

    scheduler = make_cosine_schedule(optimizer, max_lr, min_lr, warmup_steps, max_steps)

    model.train()
    train_iter = iter(train_loader)
    history    = []
    best_acc   = 0.0

    for step in range(max_steps):
        try:
            c_ids, c_mask, r_ids, r_mask = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            c_ids, c_mask, r_ids, r_mask = next(train_iter)

        c_ids = c_ids.to(device); c_mask = c_mask.to(device)
        r_ids = r_ids.to(device); r_mask = r_mask.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            chosen_r   = model(c_ids, c_mask)
            rejected_r = model(r_ids, r_mask)
            loss, metrics = reward_loss(chosen_r, rejected_r, margin=margin)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history.append(metrics)

        if step % 50 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"step {step:4d}  loss={metrics['loss']:.4f}  gap={metrics['reward_gap']:.3f}  acc={metrics['accuracy']:.2%}  lr={lr:.2e}")

        if step % eval_every == 0 and step > 0:
            model.eval()
            eval_metrics = []
            with torch.no_grad():
                for c_ids_v, c_mask_v, r_ids_v, r_mask_v in val_loader:
                    c_ids_v = c_ids_v.to(device); c_mask_v = c_mask_v.to(device)
                    r_ids_v = r_ids_v.to(device); r_mask_v = r_mask_v.to(device)
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        cr = model(c_ids_v, c_mask_v)
                        rr = model(r_ids_v, r_mask_v)
                        _, m = reward_loss(cr, rr, margin=0.0)  # <3>
                    eval_metrics.append(m)
            model.train()
            avg_acc = np.mean([m['accuracy'] for m in eval_metrics])
            avg_gap = np.mean([m['reward_gap'] for m in eval_metrics])
            print(f"  [eval] acc={avg_acc:.2%}  gap={avg_gap:.3f}")
            if avg_acc > best_acc:
                best_acc = avg_acc
                torch.save({
                    'step': step, 'model': model.state_dict(),
                    'config': config, 'acc': best_acc, 'gap': avg_gap,
                }, f'{output_dir}/reward_model_best.pt')
                print(f"  New best: acc={best_acc:.2%}")

    print(f"\nReward model training complete. Best acc: {best_acc:.2%}")
    return model, history

1. Freezing early transformer blocks (e.g. the first 4 of 6) prevents overfitting when the preference dataset is small. The early blocks capture low-level language features that are already well-learned; only the later blocks need to adapt to the preference signal.
2. Separate parameter groups: the reward head uses $10\times$ the base learning rate because it is learning from scratch, while the backbone layers already have useful representations and only need fine-tuning.
3. Eval uses `margin=0.0` so that accuracy reflects the raw preference ordering without the margin requirement artificially lowering the score.

## Evaluating the Reward Model

Preference accuracy is the primary metric, but three additional checks help diagnose failure modes.

**Reward distribution analysis.** We want the reward distributions of chosen and rejected responses to be separated, with similar variance. If both distributions have very low variance (below $0.1$), the model has not learned to discriminate. If variance is very high (above $10$), the model may be overfitting.

Implementing `analyze_reward_distribution`:

In [ ]:
@torch.no_grad()
def analyze_reward_distribution(model, dataloader, device, n_batches=50):
    """Compute stats over reward distribution. Returns dict with mean/std/accuracy."""
    model.eval()
    all_chosen, all_rejected = [], []

    for i, (c_ids, c_mask, r_ids, r_mask) in enumerate(dataloader):
        if i >= n_batches: break
        c_ids = c_ids.to(device); c_mask = c_mask.to(device)
        r_ids = r_ids.to(device); r_mask = r_mask.to(device)
        cr = model(c_ids, c_mask).cpu().float()
        rr = model(r_ids, r_mask).cpu().float()
        all_chosen.extend(cr.tolist())
        all_rejected.extend(rr.tolist())

    chosen   = np.array(all_chosen)
    rejected = np.array(all_rejected)
    all_r    = np.concatenate([chosen, rejected])

    stats = {
        'chosen_mean':   chosen.mean(),   'chosen_std':    chosen.std(),
        'rejected_mean': rejected.mean(), 'rejected_std':  rejected.std(),
        'overall_mean':  all_r.mean(),    'overall_std':   all_r.std(),
        'accuracy':      (chosen > rejected).mean(),
        'mean_gap':      (chosen - rejected).mean(),
    }

    print(f"\nReward Distribution Analysis")
    print(f"{'─'*40}")
    print(f"  Chosen   : mean={stats['chosen_mean']:+.3f}  std={stats['chosen_std']:.3f}")
    print(f"  Rejected : mean={stats['rejected_mean']:+.3f}  std={stats['rejected_std']:.3f}")
    print(f"  Gap      : mean={stats['mean_gap']:.3f}")
    print(f"  Accuracy : {stats['accuracy']:.2%}")

    if stats['overall_std'] < 0.1:
        print("  Low variance — model may not have learned to discriminate")
    if stats['overall_std'] > 10.0:
        print("  High variance — may be overfit, check for length bias")
    if stats['accuracy'] < 0.6:
        print("  Low accuracy — barely better than random")
    if stats['accuracy'] > 0.95:
        print("  Very high accuracy — may be overfitting to surface features")

    return stats

**Length bias.** A common pathology is a reward model that correlates score with response length rather than quality. We detect this by scoring responses of varying lengths with identical semantic content and computing the length-reward correlation.

Implementing `check_length_bias`:

In [ ]:
@torch.no_grad()
def check_length_bias(model, tokenizer, device, prompt="Explain photosynthesis.", lengths=None):
    """Score responses of varying lengths with identical content. Length bias shows as high correlation."""
    if lengths is None:
        lengths = [20, 50, 100, 200]

    base  = "Photosynthesis is the process by which plants convert light into energy."
    model.eval()
    print("\nLength Bias Check:")
    print(f"{'─'*50}")
    print(f"  {'Length':>8}  {'Reward':>10}  Response preview")

    scores = []
    for target_words in lengths:
        words    = (base + ' ') * (target_words // len(base.split()) + 1)
        response = ' '.join(words.split()[:target_words])
        text = (f"{SPECIAL_TOKENS['user']}\n{prompt}\n{SPECIAL_TOKENS['end']}\n"
                f"{SPECIAL_TOKENS['assistant']}\n{response}\n{SPECIAL_TOKENS['end']}")
        tokens = tokenizer.encode(text)
        ids    = torch.tensor([tokens], dtype=torch.long, device=device)
        reward = model(ids).item()
        scores.append(reward)
        print(f"  {target_words:>8}  {reward:>10.4f}  '{response[:40]}...'")

    corr = np.corrcoef(lengths, scores)[0, 1]
    print(f"\n  Length-reward correlation: {corr:.3f}")
    if corr > 0.8:
        print("  Strong length bias detected!")
    elif corr > 0.5:
        print("  Moderate length bias — monitor in downstream GRPO.")
    else:
        print("  No significant length bias.")

:::{.callout-caution}
Length bias is the most common reward model failure. Before running GRPO, always run `check_length_bias`. A reward model with strong length bias will cause GRPO to optimize verbosity rather than quality.

:::

## Best-of-N Sampling

**Best-of-N** is a simple inference-time technique: generate $N$ candidate responses, score each with the reward model, and return the one with the highest score. Response quality scales approximately as $\log N$ — each doubling of $N$ yields a roughly constant improvement. At small $N$ (e.g. $N = 8$–$16$), best-of-N is often competitive with full RLHF fine-tuning.

Implementing `best_of_n`:

In [ ]:
@torch.no_grad()
def best_of_n(policy_model, reward_model, tokenizer, prompt, n=8,
             max_new_tokens=100, temperature=0.8, device=None):
    """Generate N responses, score them all, return the best."""
    if device is None:
        device = next(policy_model.parameters()).device

    prompt_text = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n"
                   f"{SPECIAL_TOKENS['end']}\n{SPECIAL_TOKENS['assistant']}\n")
    prompt_ids  = torch.tensor([tokenizer.encode(prompt_text)],
                               dtype=torch.long, device=device)

    policy_model.eval()
    reward_model.eval()

    candidates = []
    for _ in range(n):                                         # <1>
        with torch.no_grad():
            output_ids = policy_model.generate(
                prompt_ids, max_new_tokens=max_new_tokens,
                temperature=temperature, eos_token_id=tokenizer.eos_id,
            )
        response_ids  = output_ids[0, prompt_ids.size(1):]
        response_text = tokenizer.decode(response_ids.tolist())
        score         = reward_model(output_ids).item()        # <2>
        candidates.append((response_text, score))

    candidates.sort(key=lambda x: x[1], reverse=True)         # <3>
    best_response, best_score = candidates[0]
    return best_response, best_score, candidates

1. We sample $N$ responses independently at temperature $> 0$ to get diverse candidates. Temperature $0$ (greedy) would produce $N$ identical outputs.
2. The reward model scores the full sequence (prompt + response). Using `output_ids` rather than encoding the response text separately avoids any tokenization inconsistency.
3. Sorting in descending order of score gives the best candidate at index `0`.

## Reward Calibration

When using the reward model inside GRPO, reward scores can drift over training: as the policy changes, the distribution of generated responses shifts, and the reward model may output scores in a different range than it was calibrated for. The `NormalizedRewardModel` wrapper maintains running statistics and normalizes the reward to [zero mean and unit variance]{.mark}:

In [ ]:
class NormalizedRewardModel(nn.Module):
    """Wraps RewardModel with running normalization to prevent score drift during GRPO."""

    def __init__(self, reward_model, momentum=0.99):
        super().__init__()
        self.rm       = reward_model
        self.momentum = momentum
        self.register_buffer('running_mean', torch.tensor(0.0))  # <1>
        self.register_buffer('running_var',  torch.tensor(1.0))

    def forward(self, input_ids, attention_mask=None):
        raw = self.rm(input_ids, attention_mask)
        if self.training:
            batch_mean = raw.mean().detach()
            batch_var  = raw.var().detach()
            self.running_mean = (self.momentum * self.running_mean
                                 + (1 - self.momentum) * batch_mean)  # <2>
            self.running_var  = (self.momentum * self.running_var
                                 + (1 - self.momentum) * batch_var)
        return (raw - self.running_mean) / (self.running_var.sqrt() + 1e-8)  # <3>

    def calibrate(self, dataloader, device, n_batches=100):
        """Set running stats from a calibration set. Call once before GRPO."""
        self.rm.eval()
        scores = []
        with torch.no_grad():
            for i, (c_ids, c_mask, _, _) in enumerate(dataloader):
                if i >= n_batches: break
                r = self.rm(c_ids.to(device), c_mask.to(device))
                scores.extend(r.cpu().tolist())
        scores = torch.tensor(scores)
        self.running_mean = scores.mean()
        self.running_var  = scores.var()
        print(f"Calibrated: mean={self.running_mean:.3f}  std={self.running_var.sqrt():.3f}")

1. `register_buffer` stores the running statistics as part of the module state (saved/loaded with `state_dict`) but they are not `nn.Parameter` — they do not receive gradients.
2. Exponential moving average update with momentum $0.99$: the running statistics adapt slowly to the current batch distribution, smoothing out any sudden shifts.
3. Standardization: $(r - \mu) / \sigma$. The `1e-8` prevents division by zero when variance is near $0$ at the start of training.

## Summary

**Direct Preference Optimization.**

| Concept | Key detail |
|---|---|
| RLHF objective | Maximize $\mathbb{E}[r(x,y)] - \beta\, D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}).$ |
| Optimal policy | $\pi^*(y \mid x) \propto \pi_{\text{ref}}(y \mid x) \exp(r / \beta).$ |
| DPO key step | Reward expressed in terms of optimal policy; $Z(x)$ cancels between chosen and rejected. |
| DPO loss | $-\log \sigma(\beta(\log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \log \frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}))$ |
| Reference model | Frozen copy of the SFT model. No gradients. |
| Implicit reward | $\beta \log(\pi_\theta(y \mid x) / \pi_{\text{ref}}(y \mid x)).$ |
| $\beta$ | KL penalty strength. Typical: $0.1.$ |
| Reward margin | Chosen reward minus rejected reward. Should be positive and growing. |
| Both rewards rising | Uniform divergence from reference — lower $\beta$ or LR. |
| Memory cost | Two models (policy + reference). With LoRA: policy is small; reference is full. |
| LR for DPO | Very low ($5 \times 10^{-5}$). Sensitive to LR — start low. |

: {tbl-colwidths="[30,70]"}

<br>

**Reward Model Training.**

| Concept | Key detail |
|---|---|
| Bradley-Terry model | $P(y_w \succ y_l) = \sigma(r_w - r_l).$ Logistic regression on reward gaps. |
| RM architecture | Pretrained backbone + scalar linear head. Score at last real token position. |
| Head init to zero | Neutral initial rewards. |
| Reward loss | $-\log \sigma(r_w - r_l).$ BCE with target $= 1$ and logit $=$ gap. |
| Margin loss | Requires gap $> m.$ Encourages confident separation. |
| Freeze early layers | Prevents backbone overfitting on small datasets. |
| Separate LR for head | Head learns from scratch ($10\times$ LR); backbone is pretrained ($1\times$ LR). |
| Length bias | RM correlates reward with length. Diagnose with controlled-length probes. |
| Best-of-N | Generate $N$, score all, return best. Quality scales as $\log N.$ |
| Reward normalization | Zero mean, unit variance. Prevents score drift during GRPO. |

: {tbl-colwidths="[30,70]"}

## Exercises

### DPO Exercises

1. **$\beta$ ablation.** Train the DPO model with $\beta \in \{0.01, 0.05, 0.1, 0.5\}$ and plot preference accuracy and reward margin as a function of $\beta$. At what value does the policy begin to ignore the reference entirely?

2. **Reference model drift.** In standard DPO the reference model is frozen at the SFT checkpoint. Implement "iterative DPO": after each round of DPO training, use the updated policy as the new reference and generate fresh preference data. Does iterative DPO improve preference accuracy on a held-out set?

3. **Offline vs. online data.** The toy dataset uses shuffled-word pairs as rejected responses — a very easy signal. Generate harder negatives by: (a) sampling low-temperature completions from the SFT model and rating them, or (b) using a simple rule-based quality heuristic. How does the difficulty of the negative examples affect training dynamics?

4. **Length normalization.** The log-probability $\log \pi(y \mid x)$ tends to be smaller (more negative) for longer sequences. Does this create a bias in the DPO loss? Modify `sequence_log_probs` to return the mean log-probability per token instead of the sum, and compare training curves.

5. **LoRA vs. full fine-tuning.** Train DPO with `use_lora=False` (full fine-tuning). Compare (a) final preference accuracy, (b) generation quality on a held-out prompt set, and (c) wall-clock time per step.

6. **Loss interpretation.** Show analytically that at the global optimum of $\mathcal{L}_{\text{DPO}}$, the policy $\pi_\theta$ satisfies $\pi_\theta = \pi^*$ — the optimal policy of the original KL-regularized RLHF objective.

### Reward Model Exercises

1. **Margin sensitivity.** Train the reward model with `margin` $\in \{0, 0.25, 0.5, 1.0, 2.0\}$. Plot validation accuracy and mean reward gap at the end of training for each value. Is there a clear sweet spot?

2. **Freeze depth.** Vary `freeze_layers` $\in \{0, 2, 4, 6\}$ (where $6$ is all layers frozen except the head). For a fixed 500-step budget, which setting achieves the best validation accuracy? Does the answer depend on dataset size?

3. **Reward head ablation.** The reward head is initialized to zero. Try initializing it with small random weights ($\sigma = 0.01$) or Kaiming uniform. Do either of these alternatives affect convergence speed or final accuracy?

4. **Ensemble reward model.** Train $K = 3$ reward models with different random seeds. At inference, average their scores. Does the ensemble produce better best-of-N results than a single model?

5. **Best-of-N scaling.** Run `best_of_n` with $N \in \{1, 2, 4, 8, 16, 32\}$ on a fixed set of prompts and evaluate the mean reward of the selected response. Fit a $\log N$ curve to the data. Does the empirical scaling match the theoretical prediction?

6. **Calibration.** Before GRPO training, call `NormalizedRewardModel.calibrate` on the training set. Then run a short GRPO loop. Compare training stability (variance of policy gradient estimates) with and without calibration.

■